chargement des librairies presentes dans le fichier src/imports.py

In [4]:
import sys
sys.path.append('..')
from src.imports import *

importation des modules pour le prétraitement des données : séparation et mise à l’échelle

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

chargement du dataset

In [6]:
df = pd.read_csv("../data/raw/cybersecurity_intrusion_data.csv")

Supression de la colonne User_id

In [7]:
df = df.drop(columns=["session_id"])

Suppression des valeurs manquantes 

In [8]:
df = df.dropna()
df.shape

(7571, 10)

One hot encoding

In [9]:
df = pd.get_dummies(df, columns=['protocol_type'])
df = pd.get_dummies(df, columns=['encryption_used'])
df = pd.get_dummies(df, columns=['browser_type'])

In [10]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

separation des donnees d'entrainements et de test

In [11]:
X = df.drop('attack_detected', axis = 1)
y = df['attack_detected']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
individu_prediction = X_test.iloc[5:]
y_test.iloc[5:]

5830    1
1023    0
1588    1
598     0
2514    0
       ..
7832    0
4394    0
8145    1
5854    1
1571    1
Name: attack_detected, Length: 1510, dtype: int64

In [12]:
#log transformation pour réduire l'impact des valeurs extrêmes
X_train['session_duration'] = np.log1p(X_train['session_duration'])
X_test['session_duration'] = np.log1p(X_test['session_duration'])

#normalisation des données 
cols_to_scale = ['network_packet_size', 'session_duration', 'login_attempts']
scaler = MinMaxScaler()

X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

Apercu de nos nouvelles données prétraitées

In [13]:
X_train.head()

,network_packet_size,login_attempts,session_duration,ip_reputation_score,failed_logins,unusual_time_access,protocol_type_ICMP,protocol_type_TCP,protocol_type_UDP,encryption_used_AES,encryption_used_DES,browser_type_Chrome,browser_type_Edge,browser_type_Firefox,browser_type_Safari,browser_type_Unknown
3632,0.317772,0.083333,0.603961,0.579159,4,0,0,1,0,1,0,0,0,1,0,0
2807,0.220311,0.250000,0.400460,0.494237,1,0,0,1,0,1,0,0,0,1,0,0
7569,0.261261,0.416667,0.931508,0.370030,0,1,0,1,0,1,0,0,0,1,0,0
7837,0.343980,0.000000,0.792288,0.294401,2,0,0,1,0,1,0,1,0,0,0,0
1149,0.217854,0.666667,0.650475,0.214583,2,0,0,1,0,0,1,0,1,0,0,0


Enregistrement des donnees 

In [15]:
X_train.to_csv("../data/Processed/X_train.csv", index= False)
individu_prediction.to_csv("../data/Processed/individu_prediction.csv", index= False)
X_test.to_csv("../data/Processed/X_test.csv", index= False)
y_train.to_csv("../data/Processed/y_train.csv", index= False)
y_test.to_csv("../data/Processed/y_test.csv", index= False)